In [0]:
BRONZE_SOURCE_PATH = "/Volumes/banking_fraud_platform/default/banking_fraud_platform/scored_transactions.jsonl"

bronze_df = spark.read.json(BRONZE_SOURCE_PATH)

bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "banking_fraud_platform.default.bronze_scored_transactions"
)

print(f"Bronze row count: {bronze_df.count()}")
display(bronze_df)

Bronze row count: 3702


account_age_days,account_id,amount_deviation,avg_transaction_amount_30d_customer,customer_risk_score,customer_total_transactions_30d,device_type,fraud_probability,geo_location_region,high_amount_flag,is_fraud_actual,is_high_risk_merchant_category,is_international,is_weekend,late_night_flag,merchant_category,predicted_fraud,previous_chargebacks,timestamp,transaction_amount,transaction_country,transaction_day_of_week,transaction_hour,transaction_id,transaction_type,transaction_velocity_1h,transaction_velocity_24h,velocity_ratio,weekend_international
1849,ACCAF448D3E,0.4998573466,69.1,11.1,0,tablet,0.0017044848,Africa,0,false,0,0,0,0,travel,0,0,2026-09-18T06:26:07.968543+00:00,35.04,NG,4,6,a4fdc544-1c39-4d92-86cc-1e753dc3560a,online,0,0,0.0,0
595,ACCEC46A5F8,0.5544284042,24.63,33.4,0,tablet,0.0026821729,Asia,0,false,0,0,0,0,fuel,0,0,2026-09-18T06:26:08.073263+00:00,14.21,FR,4,6,f699983c-d51f-41f9-89b6-90a72be429ce,atm_withdrawal,0,0,0.0,0
79,ACCFC177DE3,1.2152230971,98.06,2.9,0,desktop,8.555406E-4,North America,0,false,1,0,0,0,gambling,0,0,2026-09-18T06:26:08.174292+00:00,120.38,MX,4,6,b5d8ba80-1b53-44c4-9d4a-e92956e59d75,pos,0,0,0.0,0
1668,ACC53832356,0.6559974647,62.11,14.4,0,tablet,8.533697E-4,Africa,0,false,0,0,0,0,fuel,0,0,2026-09-18T06:26:08.277760+00:00,41.4,NG,4,6,bf0b2eb2-c363-4fb3-b983-bc86a4aa6324,online,0,0,0.0,0
724,ACC20738373,0.8338207466,173.39,11.5,0,desktop,0.001546502,Europe,0,false,0,0,0,0,fashion,0,0,2026-09-18T06:26:08.380043+00:00,145.41,UK,4,6,03cc8179-f99f-4155-9747-706786956e0b,atm_withdrawal,0,0,0.0,0
2304,ACCEEAAD769,1.2979391923,107.21,22.0,0,wearable,5.354288E-4,Asia,0,false,0,0,0,0,travel,0,0,2026-09-18T06:26:08.494257+00:00,140.45,AU,4,6,b7bf8564-4ea9-4008-bdf0-a07f43cd2e5b,card_present_moto,0,0,0.0,0
1923,ACC77C3CE07,1.4810323383,31.16,51.4,0,mobile,0.0591177233,Africa,0,false,0,0,0,0,fashion,0,1,2026-09-18T06:26:08.611387+00:00,47.63,IN,4,6,f8bd6238-fb42-4d00-8924-d75524f17c25,atm_withdrawal,0,0,0.0,0
2724,ACC237A6AE3,1.0859947954,83.54,39.4,0,tablet,6.496292E-4,Europe,0,false,0,0,0,0,online_services,0,0,2026-09-18T06:26:08.714613+00:00,91.81,FR,4,6,2d0c4a5f-b5bd-41f5-9fa5-99c59dec3d0e,online,0,0,0.0,0
1107,ACCC65B5216,0.6072978163,139.59,11.4,0,wearable,0.0285629854,North America,0,false,0,0,0,0,fashion,0,0,2026-09-18T06:26:08.826030+00:00,85.38,MX,4,6,835f2d48-97d1-4d77-aa82-e055f15bd716,atm_withdrawal,0,0,0.0,0
1850,ACC5B68518F,1.0281433772,141.84,2.4,0,tablet,4.7758E-5,South America,0,false,0,0,0,0,restaurant,0,0,2026-09-18T06:26:08.933334+00:00,146.86,BR,4,6,22657047-eed3-4bfe-b110-72728b2d531c,card_present_moto,0,0,0.0,0


In [0]:
from pyspark.sql.functions import col

bronze_df = spark.table("banking_fraud_platform.default.bronze_scored_transactions")
bronze_count = bronze_df.count()

# Deduplicate: keep exactly one row per transaction_id
deduped_df = bronze_df.dropDuplicates(["transaction_id"])

# Validate: drop rows with impossible values
cleaned_df = (
    deduped_df
    .filter(col("transaction_amount") > 0)
    .filter(col("account_id").isNotNull())
    .filter(col("fraud_probability").between(0, 1))
)

silver_count = cleaned_df.count()
print(f"Bronze: {bronze_count}  →  Silver: {silver_count}  (removed {bronze_count - silver_count})")

cleaned_df.write.format("delta").mode("overwrite").saveAsTable(
    "banking_fraud_platform.default.silver_scored_transactions"
)
display(cleaned_df)

Bronze: 3702  →  Silver: 3702  (removed 0)


account_age_days,account_id,amount_deviation,avg_transaction_amount_30d_customer,customer_risk_score,customer_total_transactions_30d,device_type,fraud_probability,geo_location_region,high_amount_flag,is_fraud_actual,is_high_risk_merchant_category,is_international,is_weekend,late_night_flag,merchant_category,predicted_fraud,previous_chargebacks,timestamp,transaction_amount,transaction_country,transaction_day_of_week,transaction_hour,transaction_id,transaction_type,transaction_velocity_1h,transaction_velocity_24h,velocity_ratio,weekend_international
1849,ACCAF448D3E,0.4998573466,69.1,11.1,0,tablet,0.0017044848,Africa,0,false,0,0,0,0,travel,0,0,2026-09-18T06:26:07.968543+00:00,35.04,NG,4,6,a4fdc544-1c39-4d92-86cc-1e753dc3560a,online,0,0,0.0,0
595,ACCEC46A5F8,0.5544284042,24.63,33.4,0,tablet,0.0026821729,Asia,0,false,0,0,0,0,fuel,0,0,2026-09-18T06:26:08.073263+00:00,14.21,FR,4,6,f699983c-d51f-41f9-89b6-90a72be429ce,atm_withdrawal,0,0,0.0,0
79,ACCFC177DE3,1.2152230971,98.06,2.9,0,desktop,8.555406E-4,North America,0,false,1,0,0,0,gambling,0,0,2026-09-18T06:26:08.174292+00:00,120.38,MX,4,6,b5d8ba80-1b53-44c4-9d4a-e92956e59d75,pos,0,0,0.0,0
1668,ACC53832356,0.6559974647,62.11,14.4,0,tablet,8.533697E-4,Africa,0,false,0,0,0,0,fuel,0,0,2026-09-18T06:26:08.277760+00:00,41.4,NG,4,6,bf0b2eb2-c363-4fb3-b983-bc86a4aa6324,online,0,0,0.0,0
724,ACC20738373,0.8338207466,173.39,11.5,0,desktop,0.001546502,Europe,0,false,0,0,0,0,fashion,0,0,2026-09-18T06:26:08.380043+00:00,145.41,UK,4,6,03cc8179-f99f-4155-9747-706786956e0b,atm_withdrawal,0,0,0.0,0
2304,ACCEEAAD769,1.2979391923,107.21,22.0,0,wearable,5.354288E-4,Asia,0,false,0,0,0,0,travel,0,0,2026-09-18T06:26:08.494257+00:00,140.45,AU,4,6,b7bf8564-4ea9-4008-bdf0-a07f43cd2e5b,card_present_moto,0,0,0.0,0
1923,ACC77C3CE07,1.4810323383,31.16,51.4,0,mobile,0.0591177233,Africa,0,false,0,0,0,0,fashion,0,1,2026-09-18T06:26:08.611387+00:00,47.63,IN,4,6,f8bd6238-fb42-4d00-8924-d75524f17c25,atm_withdrawal,0,0,0.0,0
2724,ACC237A6AE3,1.0859947954,83.54,39.4,0,tablet,6.496292E-4,Europe,0,false,0,0,0,0,online_services,0,0,2026-09-18T06:26:08.714613+00:00,91.81,FR,4,6,2d0c4a5f-b5bd-41f5-9fa5-99c59dec3d0e,online,0,0,0.0,0
1107,ACCC65B5216,0.6072978163,139.59,11.4,0,wearable,0.0285629854,North America,0,false,0,0,0,0,fashion,0,0,2026-09-18T06:26:08.826030+00:00,85.38,MX,4,6,835f2d48-97d1-4d77-aa82-e055f15bd716,atm_withdrawal,0,0,0.0,0
1850,ACC5B68518F,1.0281433772,141.84,2.4,0,tablet,4.7758E-5,South America,0,false,0,0,0,0,restaurant,0,0,2026-09-18T06:26:08.933334+00:00,146.86,BR,4,6,22657047-eed3-4bfe-b110-72728b2d531c,card_present_moto,0,0,0.0,0


In [0]:
from pyspark.sql.functions import count, sum as spark_sum, avg, round as spark_round, col, hour, to_timestamp

silver_df = spark.table("banking_fraud_platform.default.silver_scored_transactions")

# Gold 1: fraud by merchant category
by_category = (
    silver_df.groupBy("merchant_category")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("predicted_fraud").alias("flagged_count"),
        spark_round(avg("predicted_fraud") * 100, 2).alias("flag_rate_pct"),
        spark_round(spark_sum("transaction_amount"), 2).alias("total_amount"),
    )
    .orderBy(col("flag_rate_pct").desc())
)
by_category.write.format("delta").mode("overwrite").saveAsTable(
    "banking_fraud_platform.default.gold_fraud_by_merchant_category"
)
display(by_category)

# Gold 2: fraud by country
by_country = (
    silver_df.groupBy("transaction_country")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("predicted_fraud").alias("flagged_count"),
        spark_round(avg("predicted_fraud") * 100, 2).alias("flag_rate_pct"),
    )
    .orderBy(col("flag_rate_pct").desc())
)
by_country.write.format("delta").mode("overwrite").saveAsTable(
    "banking_fraud_platform.default.gold_fraud_by_country"
)
display(by_country)

# Gold 3: fraud by hour of day (derived from the timestamp column)
by_hour = (
    silver_df
    .withColumn("txn_hour", hour(to_timestamp(col("timestamp"))))
    .groupBy("txn_hour")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("predicted_fraud").alias("flagged_count"),
        spark_round(avg("predicted_fraud") * 100, 2).alias("flag_rate_pct"),
    )
    .orderBy("txn_hour")
)
by_hour.write.format("delta").mode("overwrite").saveAsTable(
    "banking_fraud_platform.default.gold_fraud_by_hour"
)
display(by_hour)

merchant_category,total_transactions,flagged_count,flag_rate_pct,total_amount
gambling,408,99,24.26,45654.14
luxury_goods,408,70,17.16,45166.24
electronics,352,22,6.25,42806.52
restaurant,370,22,5.95,41787.71
fuel,357,21,5.88,42042.23
digital_subscriptions,357,19,5.32,39383.73
grocery,366,16,4.37,44459.49
online_services,363,14,3.86,40209.27
fashion,373,14,3.75,42268.04
travel,348,11,3.16,40133.28


transaction_country,total_transactions,flagged_count,flag_rate_pct
UK,366,45,12.3
DE,340,39,11.47
IN,350,37,10.57
BR,399,37,9.27
MX,388,33,8.51
AU,431,33,7.66
NG,350,26,7.43
FR,339,22,6.49
US,399,24,6.02
CA,340,12,3.53


txn_hour,total_transactions,flagged_count,flag_rate_pct
5,1302,129,9.91
6,400,25,6.25
12,400,26,6.5
13,1600,128,8.0
